# dextra — جولة مشروع تطبيقية شاملة (Phases 1 → 5)

دفتر واحد يمشي بمجموعة بيانات حقيقية واحدة عبر **كل مراحل مكتبة `dextra`**
ودوالها الثماني والأربعين، كأنه مشروع علم بيانات يُعمل عليه فعلاً.

**المشروع:** بيانات اشتراكات شركة اتصالات — 600 عميل، والهدف فهم
سلوك العملاء والتنبؤ بمن سيغادر الخدمة (`churn`).

**مسار العمل:**

| المرحلة | الوحدة | الدور في المشروع |
|--------|--------|------------------|
| 1 | `stats` + `plots` | استكشاف أوّلي للبيانات الخام |
| 2 | `stats_advanced` | تحقيق إحصائي (22 دالة) |
| 3 | `cleaning` | تنظيف البيانات (10 دوال) |
| 4 | `features` | هندسة الميزات (8 دوال) |
| 5 | `selection` | اختيار الميزات (5 دوال) |

كل دالة تُستدعى بأسطر قليلة وتُخرج: جدولاً رقمياً غنياً + رسماً بصرياً
متعدّد اللوحات + جملة `Decision:` واحدة. شغّل الخلايا بالترتيب من الأعلى للأسفل.

> **المتطلبات:** `dextra` مثبّتة، ومعها `scipy` و`scikit-learn` و`plotly`
> (الأخيرتان تُستوردان كسولاً عند الحاجة).

## 0 · إعداد المشروع وتوليد البيانات الخام

نولّد مجموعة بيانات خام **واقعية بعيوبها**: أسماء أعمدة فوضوية، أنواع بيانات
خاطئة، قيم مفقودة، صفوف مكرّرة، قيم شاذة، ومخالفات منطقية — تماماً كما تصل
البيانات في أي مشروع حقيقي. هذه البيانات وحدها هي ما سيُغذّي كل المراحل.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import dextra as dx

print("dextra version:", dx.__version__)

# --- latent drivers of the telecom customer base -----------------------
rng = np.random.default_rng(2024)
N = 600

tenure  = rng.integers(1, 72, N)                       # months as a customer
plan    = rng.choice(["Basic", "Standard", "Premium"], N, p=[.45, .35, .20])
price   = {"Basic": 30.0, "Standard": 65.0, "Premium": 110.0}
monthly = np.array([price[p] for p in plan]) + rng.normal(0, 6, N)
total   = monthly * tenure + rng.normal(0, 80, N)      # total ~= monthly x tenure
age     = rng.normal(41, 13, N)
data_usage = rng.gamma(2.0, 18.0, N)                   # right-skewed usage
prev_usage = data_usage * rng.normal(0.97, 0.05, N)    # last-period usage (~duplicate)
support = rng.poisson(1.6, N)                          # support calls
city    = rng.choice(["Cairo", "Giza", "Alexandria", "Luxor"], N, p=[.4, .3, .2, .1])
contract = rng.choice(["Monthly", "Annual"], N, p=[.6, .4])
satis   = rng.choice(["low", "mid", "high"], N, p=[.25, .45, .30])
signup  = pd.to_datetime("2021-01-01") + pd.to_timedelta(rng.integers(0, 1400, N), unit="D")

# churn is driven by tenure, support calls, contract type and satisfaction
logit = (-0.04 * tenure + 0.35 * support
         + np.where(contract == "Monthly", 0.8, -0.4)
         + np.where(satis == "low", 0.9, np.where(satis == "high", -0.7, 0.0))
         + rng.normal(0, 0.6, N))
churn = (rng.random(N) < 1 / (1 + np.exp(-logit))).astype(int)

raw = pd.DataFrame({
    " Customer ID ":     [f"C{1000 + i}" for i in range(N)],   # messy column name
    "Signup Date":       signup.strftime("%Y-%m-%d"),          # stored as text
    "Age":               age,
    "Tenure (months)":   tenure,
    "Monthly Charges":   [f"{m:.2f}" for m in monthly],        # stored as text
    "Total Charges":     total,
    "Data Usage GB":     data_usage,
    "Prev Usage GB":     prev_usage,
    "Support Calls":     support,
    "City": city, "Plan": plan, "Contract": contract,
    "Satisfaction": satis, "Churn": churn,
})

# --- inject realistic data-quality problems ----------------------------
raw.loc[rng.choice(N, 45, replace=False), "Age"] = np.nan            # missing
raw.loc[rng.choice(N, 30, replace=False), "Total Charges"] = np.nan  # missing
raw.loc[rng.choice(N, 8,  replace=False), "Data Usage GB"] = rng.uniform(300, 600, 8)  # outliers
raw.loc[rng.choice(N, 4,  replace=False), "Age"] = rng.uniform(-5, 0, 4)   # impossible age
raw.loc[rng.choice(N, 3,  replace=False), "Age"] = rng.uniform(125, 150, 3)# impossible age
raw = pd.concat([raw, raw.iloc[rng.choice(N, 12, replace=False)]],         # duplicate rows
                ignore_index=True)
raw["City"] = raw["City"].where(rng.random(len(raw)) > 0.05, raw["City"] + "  ")  # stray spaces

print("raw dataset shape:", raw.shape)

In [ ]:
# A first glance at the raw data
raw.head(8)

In [ ]:
raw.info()

## المرحلة 1 · الاستكشاف الأوّلي  —  `stats` + `plots`

أول ما يفعله أي محلّل: ينظر إلى البيانات. ثلاث دوال أساسية تعطي ملخصاً رقمياً
غنياً ورسمين بصريين بسطر واحد لكلٍّ منها.

In [ ]:
# describe_numeric: a dense numeric summary (mean, std, quartiles, skew, outliers...)
dx.describe_numeric(raw)

In [ ]:
# plot_histograms: distribution of every numeric column + an adjacent stats panel
dx.plot_histograms(raw)

In [ ]:
# plot_boxplots: interactive (plotly) boxplots highlighting outliers
dx.plot_boxplots(raw)

## المرحلة 2 · التحقيق الإحصائي  —  `stats_advanced` (22 دالة)

نُجري تحقيقاً إحصائياً على البيانات **الخام** لفهم طبيعتها وعلاقاتها قبل
التنظيف: التوزيعات، الارتباطات، فترات الثقة، اختبارات الفرضيات، وتشخيصات
ما قبل النمذجة.

### 2.1 · إحصاءات وصفية موسّعة

In [ ]:
# z_scores: standardised scores + flagging of extreme values
dx.z_scores(raw)

In [ ]:
# pearson_skewness: direction and strength of distribution asymmetry
dx.pearson_skewness(raw)

In [ ]:
# empirical_rule_check: does each column obey the 68-95-99.7 rule?
dx.empirical_rule_check(raw)

In [ ]:
# outliers_report: outlier counts and bounds (IQR method)
dx.outliers_report(raw)

### 2.2 · التحليل ثنائي المتغيّر

In [ ]:
# correlation_matrix: pairwise correlations with significance
dx.correlation_matrix(raw)

In [ ]:
# simple_linear_regression: fit, coefficients, R^2 and a fitted-line plot
dx.simple_linear_regression(raw, x="Tenure (months)", y="Total Charges")

### 2.3 · أدوات استكشاف السوق / البيانات

In [ ]:
# missing_report: where are the gaps, and how big?
dx.missing_report(raw)

In [ ]:
# frequency_table: category counts and proportions for a column
dx.frequency_table(raw, col="City")

In [ ]:
# cross_tab: contingency table of two categoricals + chi-square
dx.cross_tab(raw, row="City", col="Churn")

In [ ]:
# group_compare: compare numeric columns across the levels of a group
dx.group_compare(raw, group_col="Plan", value_cols=["Total Charges", "Age"])

### 2.4 · الاستدلال الإحصائي

In [ ]:
# confidence_interval_mean: 95% CI for the mean customer age
dx.confidence_interval_mean(raw["Age"].dropna())

In [ ]:
# confidence_interval_proportion: 95% CI for the churn rate
dx.confidence_interval_proportion(successes=int(raw["Churn"].sum()), n=len(raw))

In [ ]:
# sample-size planning: how many customers to survey for a target precision
dx.sample_size_mean(margin_error=2.0, std=float(raw["Total Charges"].std()))
dx.sample_size_proportion(margin_error=0.05, p=float(raw["Churn"].mean()))

### 2.5 · اختبارات الفرضيات

In [ ]:
# normality_test: is data usage normally distributed?
dx.normality_test(raw["Data Usage GB"])

In [ ]:
# t_test_one_sample: is mean age different from 40?
dx.t_test_one_sample(raw["Age"].dropna(), popmean=40)

In [ ]:
# t_test_two_sample: do churned vs retained customers differ in total charges?
dx.t_test_two_sample(raw.loc[raw["Churn"] == 1, "Total Charges"].dropna(),
                     raw.loc[raw["Churn"] == 0, "Total Charges"].dropna())

In [ ]:
# t_test_paired: did usage change from the previous period to now?
dx.t_test_paired(raw["Prev Usage GB"], raw["Data Usage GB"])

In [ ]:
# anova_oneway: does total charge differ across the three plans?
dx.anova_oneway(raw, group_col="Plan", value_col="Total Charges")

In [ ]:
# chi_square_independence: is churn independent of city?
dx.chi_square_independence(raw, row="City", col="Churn")

### 2.6 · تشخيصات ما قبل النمذجة

In [ ]:
# vif_scores: multicollinearity among numeric predictors
dx.vif_scores(raw)

In [ ]:
# class_imbalance: how balanced is the churn target?
dx.class_imbalance(raw["Churn"])

## المرحلة 3 · تنظيف البيانات  —  `cleaning` (10 دوال)

التحقيق كشف العيوب؛ الآن نُصلحها عبر مراحل DAMA الثماني. كل دالة تُرجع
DataFrame **جديداً** (الأصل لا يتغيّر)، فنُسلسلها: من `raw` الفوضوية إلى
`clean_df` الجاهزة. ثلاث دوال **فاحصة** تَعرض المشكلة، وسبع **فاعلة** تُصلحها.

In [ ]:
# clean_report: one master audit of every quality dimension
dx.clean_report(raw)

In [ ]:
# na_show / dup_show / out_show: focused inspectors (they only diagnose)
dx.na_show(raw)

In [ ]:
dx.dup_show(raw)

In [ ]:
dx.out_show(raw)

**التنظيف يبدأ** — نُسلسل الدوال الفاعلة، ونلتقط ناتج كلٍّ بـ `return_df=True`:

In [ ]:
# 1) standardize_columns: fix the messy column names (explicit name_map)
name_map = {
    " Customer ID ": "customer_id", "Signup Date": "signup_date",
    "Age": "age", "Tenure (months)": "tenure_months",
    "Monthly Charges": "monthly_charges", "Total Charges": "total_charges",
    "Data Usage GB": "data_usage_gb", "Prev Usage GB": "prev_usage_gb",
    "Support Calls": "support_calls", "City": "city", "Plan": "plan",
    "Contract": "contract", "Satisfaction": "satisfaction", "Churn": "churn",
}
df_std = dx.standardize_columns(raw, name_map=name_map, return_df=True)

In [ ]:
# 2) cast_types: fix wrong dtypes (text -> number / datetime)
df_cast = dx.cast_types(
    df_std,
    schema={"monthly_charges": "float64", "signup_date": "datetime64[ns]"},
    return_df=True)

In [ ]:
# 3) validate_rules: check business / consistency rules (reports only)
rules = [
    {"name": "age_valid",            "check": "age.between(0, 120)"},
    {"name": "charges_non_negative", "check": "total_charges >= 0"},
    {"name": "tenure_positive",      "check": "tenure_months >= 1"},
]
dx.validate_rules(df_cast, rules)

In [ ]:
# 4) handle_missing: fill the gaps (auto per-column strategy)
df_filled = dx.handle_missing(df_cast, return_df=True)

In [ ]:
# 5) dedupe: drop the duplicated rows
df_dedup = dx.dedupe(df_filled, return_df=True)

In [ ]:
# 6) clip_outliers: tame the extreme values
clean_df = dx.clip_outliers(
    df_dedup, cols=["data_usage_gb", "age", "total_charges"], return_df=True)

In [ ]:
# the cleaned dataset -- re-audit to confirm it is healthy now
print("clean_df shape:", clean_df.shape, "| remaining nulls:",
      int(clean_df.isna().sum().sum()))
dx.clean_report(clean_df)

## المرحلة 4 · هندسة الميزات  —  `features` (8 دوال)

البيانات نظيفة؛ الآن نحوّلها إلى إشارات تنفع النموذج. **القاعدة الذهبية:**
نقسم البيانات أولاً (train/test)، ثم نُدرّب كل تحويل على التدريب
(`return_params=True`) ونُطبّق نفس المعاملات على الاختبار (`params=...`)
بلا إعادة تدريب — هذا هو الضمان ضد تسرّب البيانات.

In [ ]:
# the inviolable boundary: split BEFORE any feature engineering
train_df = clean_df.iloc[:480].reset_index(drop=True)
test_df  = clean_df.iloc[480:].reset_index(drop=True)
print("train:", train_df.shape, " test:", test_df.shape)

In [ ]:
# transform: reshape the skewed data-usage distribution (log1p)
train_t, p_transform = dx.transform(train_df, cols=["data_usage_gb"],
                                    method="log1p", return_params=True)
test_t = dx.transform(test_df, params=p_transform)        # apply, no re-fit

In [ ]:
# scale: put numeric columns on a common scale (z-score)
train_s, p_scale = dx.scale(train_df, cols=["monthly_charges", "total_charges", "age"],
                            method="standard", return_params=True)
test_s = dx.scale(test_df, params=p_scale)

In [ ]:
# bin: discretise age into 4 quantile bins
train_b, p_bin = dx.bin(train_df, cols=["age"], method="quantile",
                        n_bins=4, return_params=True)
test_b = dx.bin(test_df, params=p_bin)

In [ ]:
# encode: turn the 'city' category into numbers (one-hot)
train_e, p_enc = dx.encode(train_df, cols=["city"], method="onehot",
                           return_params=True)
test_e = dx.encode(test_df, params=p_enc)

In [ ]:
# dtfeats: extract calendar + cyclical signals from the signup date
train_d, p_dt = dx.dtfeats(train_df, cols=["signup_date"], method="both",
                           return_params=True)
test_d = dx.dtfeats(test_df, params=p_dt)

In [ ]:
# cross: an interaction feature -- total charge per month of tenure
train_c, p_cross = dx.cross(train_df, pairs=[("total_charges", "tenure_months")],
                            method="ratio", return_params=True)
test_c = dx.cross(test_df, params=p_cross)

In [ ]:
# aggfeat: average monthly charge per city, leakage-safe (expanding as_of window)
train_a, p_agg = dx.aggfeat(train_df, group="city", value="monthly_charges",
                            agg="mean", as_of="signup_date", return_params=True)
test_a = dx.aggfeat(test_df, params=p_agg)

In [ ]:
# featpipe: chain several feature steps into one saveable pipeline
fe_recipe = [
    {"fn": "transform", "cols": ["data_usage_gb"], "method": "log1p"},
    {"fn": "scale", "cols": ["monthly_charges", "total_charges"], "method": "standard"},
    {"fn": "encode", "cols": ["plan"], "method": "onehot"},
]
train_fe, fe_params = dx.featpipe(train_df, steps=fe_recipe, return_params=True)
test_fe = dx.featpipe(test_df, params=fe_params)          # apply the whole pipeline
print("engineered train:", train_fe.shape, " -> test:", test_fe.shape)

## المرحلة 5 · اختيار الميزات  —  `selection` (5 دوال)

هندسة الميزات أنتجت إشارات كثيرة؛ الآن نُبقي ما ينفع النموذج فعلاً ونحذف
الزائد. نفس عقد fit/apply: القرار يُتعلَّم على التدريب ويُطبَّق حرفياً على
الاختبار. (الدوال القائمة على النماذج تحتاج `scikit-learn`.)

In [ ]:
# the candidate numeric features and the target
feat = ["age", "tenure_months", "monthly_charges", "total_charges",
        "data_usage_gb", "prev_usage_gb", "support_calls"]

In [ ]:
# redundancy: drop features that duplicate information (correlation filter)
train_red, p_red = dx.redundancy(train_df, cols=feat, method="correlation",
                                 threshold=0.9, return_params=True)

In [ ]:
# relevance: rank features by their univariate link to churn (ANOVA F-test)
train_rel, p_rel = dx.relevance(train_df, y="churn", cols=feat,
                                method="anova", keep=5, return_params=True)

In [ ]:
# importance: rank features by a random-forest's importances
train_imp, p_imp = dx.importance(train_df, y="churn", cols=feat,
                                 method="tree", keep=5, return_params=True)

In [ ]:
# rfe: recursive feature elimination down to the best 4 features
train_rfe, p_rfe = dx.rfe(train_df, y="churn", cols=feat, keep=4,
                          estimator="tree", return_params=True)

In [ ]:
# selectpipe: chain selectors into one pipeline, then apply to held-out data
sel_recipe = [
    {"fn": "redundancy", "method": "correlation", "threshold": 0.9},
    {"fn": "relevance",  "method": "anova", "keep": 5},
    {"fn": "importance", "method": "tree",  "keep": 3},
]
sel_train, sel_params = dx.selectpipe(train_df, steps=sel_recipe, y="churn",
                                      return_params=True)
sel_test = dx.selectpipe(test_df, params=sel_params)      # apply, no re-score
print("final selected columns:", list(sel_train.columns))

## الخلاصة

مشينا بمجموعة بيانات واحدة عبر مكتبة `dextra` كاملة: استكشفناها، حقّقنا فيها
إحصائياً، نظّفناها، هندسنا ميزاتها، ثم اخترنا أفضلها — كلها بأسطر قليلة وعقد
موحّد آمن ضد تسرّب البيانات.

**ملاحظة عن الأسماء المختصرة:** لكل دالة طويلة alias قصير
(`describe_numeric`→`numdesc`، `z_scores`→`zsc`، `handle_missing`→`na_fix`،
`redundancy`→`redun`، `importance`→`imps` ...). الخلية التالية تتأكّد من ذلك.

In [ ]:
# every long-named function also has a short alias -- same object
checks = {
    "numdesc -> describe_numeric": dx.numdesc is dx.describe_numeric,
    "zsc -> z_scores":             dx.zsc is dx.z_scores,
    "na_fix -> handle_missing":    dx.na_fix is dx.handle_missing,
    "winsor -> clip_outliers":     dx.winsor is dx.clip_outliers,
    "redun -> redundancy":         dx.redun is dx.redundancy,
    "imps -> importance":          dx.imps is dx.importance,
}
for name, ok in checks.items():
    print(("OK  " if ok else "FAIL") + "  " + name)

In [ ]:
print("=" * 60)
print("  dextra full walkthrough complete")
print("  Phase 1  exploration .......... 3 functions")
print("  Phase 2  statistics ........... 22 functions")
print("  Phase 3  cleaning ............. 10 functions")
print("  Phase 4  feature engineering .. 8 functions")
print("  Phase 5  feature selection .... 5 functions")
print("  total ......................... 48 functions exercised")
print("=" * 60)